# Phase 3 - Rhetorical roles with Arch 1 - GPT-2 style decoder (word-level tokenizer)

Starts from the Phase-1 model (12-layer GPT-2 style decoder (111M), learned positions, word-level lowercased vocabulary.) and labels every sentence of a judgment with one of 13 rhetorical
roles. Primary metric: macro-F1. CL and IT documents are trained together; a domain embedding tells them apart and
scores are reported per domain.

**The model**
1. The Phase-1 backbone reads *packs* of consecutive sentences (up to 512 tokens), so each sentence is encoded next to
   its neighbours (Cohan et al. 2019, *Pretrained LMs for sequential sentence classification*). For the decoder-only
   models the causal mask is lifted (Dukic & Snajder 2024; LLM2Vec). Each sentence is mean-pooled.
2. Relative-position and domain embeddings are added: the role depends strongly on where a sentence sits.
3. A BiLSTM reads the whole document's sentence vectors, then a CRF picks the label sequence - roles come in long blocks
   (94% of CL and 75% of IT sentences repeat the previous label). (Bhattacharya et al. 2019; Kalamkar et al. 2022.)
4. Auxiliary *label-shift* head: "does the role change at this sentence?" (Malik et al. 2022, the paper behind this dataset).
5. Macro-F1 rewards rare roles (Issue, None, RulingByPresentCourt...): class-weighted loss, and at the end a prior
   correction (logit adjustment, Menon et al. 2021) whose strength - and CRF vs argmax - is chosen on the dev set.

**Experiments** (each one is its own submission)

| Experiment | Trained parameters |
|---|---|
| `full` | everything, with layer-wise LR decay - strong reference and the distillation teacher |
| `lorma` | **LoRMA** (compulsory): `(alpha/r BA + I) W x` on the attention projections (Bihany, Patel & Modi 2025) |
| `lora` | LoRA on the same projections (Hu et al. 2021) |
| `ia3` | IA3: learned rescaling of keys/values and feed-forward activations (Liu et al. 2022) |
| `prompt` | prompt tuning: 16 learned virtual tokens in front of every pack (Lester et al. 2021) |
| `kd` | distillation: the fine-tuned `full` model teaches a half-depth student through soft labels **and** its sentence representations (Hinton et al. 2015) |

In every PEFT run the backbone is frozen; only the adapter parameters and the task head (BiLSTM, CRF, embeddings) train.

**Kaggle settings:** GPU T4 x2, Internet on, secret `HF` holding your Hugging Face token. Then Run All.

## 1. Setup

In [ ]:
import os, sys, json, glob, shutil, time
import torch
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

WORK_DIR = "/kaggle/working/phase3_arch1"
os.makedirs(WORK_DIR, exist_ok=True)
sys.path.insert(0, WORK_DIR)

print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## 2. Configuration
Everything you might want to change lives here.

In [ ]:
%%writefile /kaggle/working/phase3_arch1/config.py
"""Every knob of this Phase-3 (rhetorical role) notebook."""
import os
import json

ARCH = "arch1"
WORK_DIR = f"/kaggle/working/phase3_{ARCH}"
DATA_DIR = os.path.join(WORK_DIR, "data")
CACHE_DIR = os.path.join(WORK_DIR, "phase1")        # Phase-1 files downloaded from Hugging Face land here
OUT_DIR = os.path.join(WORK_DIR, "runs")             # one folder per experiment

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
_P3 = os.path.join(DATA_DIR, "Phase_3_RR")
FILES = {f"{dom}_{split}": os.path.join(_P3, "Test" if split == "test" else "Train", f"{dom}_{split}.json")
         for dom in ("CL", "IT") for split in ("train", "dev", "test")}

# ------------------------------------------------------------------ the Phase-1 model
# Looked up in this order: an attached Kaggle dataset whose path mentions "arch1", then these HF repos.
PHASE1_REPOS = ["D-240357/legal-gpt2-decoder"]
WEIGHT_FILES = ["model_state_dict.pt"]
CONFIG_FILES = ["config.json"]
TOKENIZER_FILES = ["custom_tokenizer.json"]
TOKENIZER_REPOS = PHASE1_REPOS
KAGGLE_HINT = "arch1"

RR_REPO = "D-240357/legal-rr-arch1"             # fine-tuned checkpoints, results and submission zips go here

# ------------------------------------------------------------------ experiments
# Each one is a separate submission. "full" is the strong reference and the distillation teacher;
# "kd" waits for it to finish.
METHODS = ["full", "lorma", "lora", "ia3", "prompt", "kd"]
KD_TEACHER = "full"

# ------------------------------------------------------------------ input / context
MAX_SENT_TOKENS = 96           # longer sentences are cut (median sentence is ~40 tokens)
PACK_TOKENS = 512              # consecutive sentences share one sequence, so each one is read in context
PACK_CONTEXT = True            # False = every sentence encoded alone (the "sentence-only" ablation)
DOC_CONTEXT = "bilstm"         # "none" = no document-level layer (another ablation)
BIDIRECTIONAL = True            # lift the causal mask while fine-tuning (Dukic & Snajder 2024; LLM2Vec)
SPAN_SENTS = 96               # consecutive sentences per training example (whole documents at test time)
TRAIN_ON_DEV = False           # True for a final run on train+dev once the settings are chosen
TRAIN_DOC_LIMIT = None         # set to e.g. 8 for a quick end-to-end smoke test

# ------------------------------------------------------------------ head and losses
CONTEXT_DIM = 512
HEAD_DROPOUT = 0.2
CE_WEIGHT = 0.5                # class-weighted cross-entropy next to the CRF loss (helps the rare roles)
CLASS_WEIGHT_POWER = 0.5       # weight = (1 / frequency) ** power
SHIFT_WEIGHT = 0.3             # auxiliary "does the role change here?" loss (Malik et al. 2022)
TAU_GRID = [0.0, 0.25, 0.5, 0.75, 1.0]   # prior correction tried on dev at the end (Menon et al. 2021)

# ------------------------------------------------------------------ optimisation
EPOCHS = 20                    # Phase-2 runs peaked in their last epochs, and one epoch here is only ~1-2 min
PATIENCE = 5
GRAD_ACCUM = 2
EVAL_PACK_BATCH = 16
WARMUP_FRAC = 0.1
WEIGHT_DECAY = 0.01
HEAD_LR = 1e-3
FULL_LR = 1e-4                 # top layer; each layer below gets LLRD times the one above it
                               # (in Phase 2 a higher LR on the top layers beat a timid full fine-tune)
LLRD = 0.85
ADAPTER_LR = 5e-4
ADAPTER_RANK = 8
LORA_ALPHA = 16
LORMA_ALPHA = 8
ADAPTER_DROPOUT = 0.05
ADAPTER_TARGETS = ["W_qkv", "W_o"]
IA3_OUT_TARGETS = ["W_qkv"]       # key/value projections
IA3_IN_TARGETS = ["down"]         # feed-forward down projections
PROMPT_LEN = 16
PROMPT_LR = 1e-2

KD_TEMPERATURE = 2.0
KD_ALPHA = 0.5                 # weight of the task loss; the rest goes to matching the teacher
KD_HIDDEN_WEIGHT = 1.0         # also match the teacher's sentence representations
STUDENT_LAYERS = [0, 2, 4, 6, 8, 10]

SEED = 13

_override = os.environ.get("P3_CFG_OVERRIDE")
if _override:
    globals().update(json.loads(_override))

## 3. Code
The Phase-1 tokenizer and model (unchanged, so the trained weights load), plus the rhetorical-role pieces.

In [ ]:
%%writefile /kaggle/working/phase3_arch1/tokenizer.py
"""Arch 1's word-level tokenizer (exactly as in Phase 1) plus an offset-aware encoder for NER."""
import re
import json
from collections import Counter


class CustomTokenizer:
    def __init__(self, vocab_size=32000):
        self.vocab_size = vocab_size
        self.special_tokens = ["<pad>", "<unk>", "<bos>", "<eos>"]
        self.word2id = {w: i for i, w in enumerate(self.special_tokens)}
        self.id2word = {i: w for i, w in enumerate(self.special_tokens)}
        self.pad_id = 0
        self.unk_id = 1
        self.bos_id = 2
        self.eos_id = 3
        self.counter = Counter()

    def _tokenize_text(self, text):
        return re.findall(r"\w+|[^\w\s]", text, re.UNICODE)

    def encode(self, text, add_special_tokens=False):
        ids = [self.word2id.get(tok, self.unk_id) for tok in self._tokenize_text(text.lower())]
        if add_special_tokens:
            ids = [self.bos_id] + ids + [self.eos_id]
        return ids

    def load(self, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        self.word2id = data["word2id"]
        self.id2word = {int(k): v for k, v in data["id2word"].items()}
        self.vocab_size = len(self.word2id)


# Phase 1 used \w+ which glues letters and digits ("Pradesh1", "ENT08"). Splitting them lifts the share of
# gold spans that start and end on a token boundary; both halves are ordinary vocabulary words.
NER_WORD_RE = re.compile(r"[^\W\d_]+|\d+|_+|[^\w\s]", re.UNICODE)


class NerTokenizer(CustomTokenizer):
    split_digits = True

    def encode_with_offsets(self, text):
        splitter = NER_WORD_RE if self.split_digits else re.compile(r"\w+|[^\w\s]", re.UNICODE)
        ids, starts, ends = [], [], []
        for m in splitter.finditer(text):
            ids.append(self.word2id.get(m.group().lower(), self.unk_id))
            starts.append(m.start())
            ends.append(m.end())
        return ids, starts, ends

In [ ]:
%%writefile /kaggle/working/phase3_arch1/phase1_model.py
"""
Arch 1 (GPT-2 style decoder) exactly as trained in Phase 1, so the saved weights load unchanged.
Phase 2 adds one switch: `causal`, so attention can look right as well as left while fine-tuning for NER.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_head, dropout, max_len):
        super().__init__()
        self.n_head = n_head
        self.head_dim = d_model // n_head
        self.W_qkv = nn.Linear(d_model, 3 * d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.resid_drop = nn.Dropout(dropout)
        self.dropout_p = dropout
        self.causal = True

    def forward(self, x, key_mask=None):
        B, T, D = x.shape
        qkv = self.W_qkv(x).reshape(B, T, 3, self.n_head, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        p = self.dropout_p if self.training else 0.0
        if key_mask is not None:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=key_mask, dropout_p=p)
        else:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=self.causal, dropout_p=p)
        out = out.permute(0, 2, 1, 3).reshape(B, T, D)
        return self.resid_drop(self.W_o(out))


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.up = nn.Linear(d_model, d_ff)
        self.down = nn.Linear(d_ff, d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.drop(self.down(F.gelu(self.up(x))))


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_head, d_ff, dropout, max_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_head, dropout, max_len)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout)

    def forward(self, x, key_mask=None):
        x = x + self.attn(self.ln1(x), key_mask)
        x = x + self.ffn(self.ln2(x))
        return x


class GPT2LM(nn.Module):
    def __init__(self, vocab_size, d_model, n_head, n_layer, d_ff, dropout, max_len):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.emb_drop = nn.Dropout(dropout)
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, n_head, d_ff, dropout, max_len)
            for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.token_emb.weight

    def forward(self, x):
        B, T = x.shape
        positions = torch.arange(T, device=x.device).unsqueeze(0)
        h = self.emb_drop(self.token_emb(x) + self.pos_emb(positions))
        for layer in self.layers:
            h = layer(h)
        return self.head(self.ln_f(h))

In [ ]:
%%writefile /kaggle/working/phase3_arch1/hub_files.py
"""Finds Phase-1 files: an attached Kaggle dataset first, then a local cache, then Hugging Face."""
import os
import re

from huggingface_hub import hf_hub_download

KAGGLE_INPUT = "/kaggle/input"


def _squash(text):
    return re.sub(r"[^a-z0-9]", "", text.lower())


def find_in_kaggle_input(filename, hint):
    """Only accepts a file whose path mentions the architecture (e.g. 'arch3', 'arch-3', 'Arch_3'),
    because arch 2, 3 and 4 all call their weights model_final.pt."""
    if not os.path.isdir(KAGGLE_INPUT):
        return None
    hits = []
    for root, _, files in os.walk(KAGGLE_INPUT):
        if filename in files:
            path = os.path.join(root, filename)
            if _squash(hint) in _squash(path):
                hits.append(path)
    return sorted(hits)[0] if hits else None


def fetch_from_hub(repo, filename, local_dir):
    try:
        return hf_hub_download(repo, filename, local_dir=local_dir, token=os.environ.get("HF_TOKEN"))
    except Exception:
        return None


def locate(filenames, repos, hint, local_dir):
    """Names are tried in order of preference. Returns (path, where it came from)."""
    for name in filenames:
        path = find_in_kaggle_input(os.path.basename(name), hint)
        if path:
            return path, "kaggle input"
    for name in filenames:
        for repo in repos:
            cached = os.path.join(local_dir, _squash(repo), name)
            if os.path.exists(cached):
                return cached, f"cache of hf://{repo}"
            path = fetch_from_hub(repo, name, os.path.join(local_dir, _squash(repo)))
            if path:
                return path, f"hf://{repo}"
    raise FileNotFoundError(f"None of {filenames} found under /kaggle/input (path must mention '{hint}') or in "
                            f"{repos}. Attach the Phase-1 weights as a Kaggle dataset, or check HF_TOKEN and the repo names.")

In [ ]:
%%writefile /kaggle/working/phase3_arch1/backbone.py
"""Loads the Phase-1 arch 1 weights and exposes them as a sentence encoder for rhetorical roles."""
import json
import os

import torch
import torch.nn as nn

import config as C
from hub_files import locate, fetch_from_hub  # noqa: F401  (fetch_from_hub is used by run_rr)
from phase1_model import GPT2LM
from tokenizer import NerTokenizer


def add_prompt(x, mask, prompt):
    """Prompt tuning (Lester et al. 2021): prepend learned vectors to the token embeddings."""
    if prompt is None:
        return x, mask, 0
    P = prompt.size(0)
    x = torch.cat([prompt.to(x.dtype).unsqueeze(0).expand(x.size(0), -1, -1), x], dim=1)
    mask = torch.cat([mask.new_ones(mask.size(0), P), mask], dim=1)
    return x, mask, P


def _find(names, repos=None):
    return locate(names, repos or C.PHASE1_REPOS, C.KAGGLE_HINT, C.CACHE_DIR)


def load_tokenizer():
    path, _ = _find(C.TOKENIZER_FILES, C.TOKENIZER_REPOS)
    tok = NerTokenizer()
    tok.load(path)
    tok.sep_id = tok.eos_id               # marks the start of every sentence inside a pack
    return tok


class Backbone(nn.Module):
    def __init__(self, gpt, keep_layers, bidirectional):
        super().__init__()
        self.d_model = gpt.token_emb.embedding_dim
        self.tok_emb = gpt.token_emb
        self.pos_emb = gpt.pos_emb
        self.emb_drop = gpt.emb_drop
        blocks = gpt.layers if keep_layers is None else [gpt.layers[i] for i in keep_layers]
        self.layers = nn.ModuleList(blocks)
        self.final_norm = gpt.ln_f
        self.bidirectional = bidirectional
        for blk in self.layers:
            blk.attn.causal = not bidirectional

    def forward(self, ids, mask, prompt=None):
        h, mask, P = add_prompt(self.tok_emb(ids), mask, prompt)
        pos = torch.arange(h.size(1), device=ids.device).unsqueeze(0)
        h = self.emb_drop(h + self.pos_emb(pos))
        key_mask = mask[:, None, None, :] if (self.bidirectional and not bool(mask.all())) else None
        for blk in self.layers:
            h = blk(h, key_mask)
        return self.final_norm(h)[:, P:]


def build_backbone(keep_layers=None, bidirectional=True):
    cfg_path, _ = _find(C.CONFIG_FILES)
    weights_path, source = _find(C.WEIGHT_FILES)
    with open(cfg_path) as f:
        cfg = json.load(f)
    gpt = GPT2LM(cfg["vocab_size"], cfg["d_model"], cfg["n_head"], cfg["n_layer"], cfg["d_ff"],
                 cfg["dropout"], cfg["max_len"])
    state = torch.load(weights_path, map_location="cpu")
    state = {k.replace("_orig_mod.", ""): v.float() for k, v in state.items()}
    gpt.load_state_dict(state)
    if os.environ.get("P2_VERBOSE", "1") == "1":
        print(f"Phase-1 arch 1 weights: {weights_path} ({source})", flush=True)
    return Backbone(gpt, keep_layers, bidirectional)


def student_layers():
    return list(C.STUDENT_LAYERS)

In [ ]:
%%writefile /kaggle/working/phase3_arch1/rr_core.py
"""
Phase 3 (rhetorical roles) machinery shared by all four architectures.

Model, bottom to top:
  1. the Phase-1 backbone reads packs of consecutive sentences (up to PACK_TOKENS), so every sentence is encoded
     next to its neighbours (Cohan et al. 2019), and each sentence is mean-pooled;
  2. relative position + domain (CL / IT) embeddings are added;
  3. a BiLSTM runs over the whole document's sentence vectors;
  4. a linear-chain CRF picks the label sequence (roles come in long blocks: 94% of CL and 75% of IT sentences
     share the previous sentence's label);
  5. an auxiliary head predicts whether the role changes at each sentence (label-shift prediction, Malik et al. 2022).
"""
import json
import math
import random
import zipfile
import os

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

LABELS = ["Fact", "Issue", "ArgumentPetitioner", "ArgumentRespondent", "Statute", "Dissent", "PrecedentReliedUpon",
          "PrecedentNotReliedUpon", "PrecedentOverruled", "RulingByLowerCourt", "RatioOfTheDecision",
          "RulingByPresentCourt", "None"]
LABEL_ID = {l: i for i, l in enumerate(LABELS)}
N_LABELS = len(LABELS)
DOMAINS = ["CL", "IT"]
N_POS_BUCKETS = 20


# --------------------------------------------------------------------------------------------- data
def load_split(path, domain):
    with open(path, encoding="utf-8") as f:
        raw = json.load(f)
    docs = []
    if isinstance(raw, dict):                                   # test files: {doc_id: [sentences]}
        raw = [{"id": k, "text": v} for k, v in raw.items()]
    for d in raw:
        labels = [LABEL_ID[l] for l in d["labels"]] if "labels" in d else None
        docs.append({"id": str(d["id"]), "domain": DOMAINS.index(domain), "sents": list(d["text"]), "labels": labels})
    return docs


def tokenize_docs(docs, tok, max_sent_tokens):
    """Every sentence becomes [separator] + its first max_sent_tokens tokens."""
    for d in docs:
        d["tok"] = [[tok.sep_id] + tok.encode(s)[:max_sent_tokens] for s in d["sents"]]
    return docs


def pack(sent_tokens, pack_tokens):
    """Greedily put consecutive sentences into sequences of at most pack_tokens tokens.
    Returns the packs and, for every sentence, (pack index, first token, last token + 1)."""
    packs, where, cur = [], [], []
    for toks in sent_tokens:
        if cur and len(cur) + len(toks) > pack_tokens:
            packs.append(cur)
            cur = []
        where.append((len(packs), len(cur), len(cur) + len(toks)))
        cur = cur + toks
    if cur:
        packs.append(cur)
    return packs, where


def batch_packs(packs, where, pad_id, device, one_per_sentence=False):
    if one_per_sentence:                                        # sentence-only ablation: no neighbours in view
        packs = [packs[p][a:b] for p, a, b in where]
        where = [(i, 0, b - a) for i, (_, a, b) in enumerate(where)]
    L = max(len(p) for p in packs)
    ids = torch.full((len(packs), L), pad_id, dtype=torch.long)
    mask = torch.zeros((len(packs), L), dtype=torch.bool)
    sent_of_token = torch.full((len(packs), L), -1, dtype=torch.long)
    for i, p in enumerate(packs):
        ids[i, :len(p)] = torch.tensor(p)
        mask[i, :len(p)] = True
    for s, (p, a, b) in enumerate(where):
        sent_of_token[p, a:b] = s
    return ids.to(device), mask.to(device), sent_of_token.to(device)


def position_buckets(n):
    return torch.tensor([min(N_POS_BUCKETS - 1, int(N_POS_BUCKETS * i / n)) for i in range(n)], dtype=torch.long)


# ----------------------------------------------------------------------------------- adapters
class LoRALinear(nn.Module):
    """Hu et al. 2021: W x + (alpha / r) B A x, B = 0 at the start."""

    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale


class LoRMALinear(nn.Module):
    """Bihany, Patel & Modi 2025 (LoRMA+): h = (alpha/r B A + I) W0 x, computed as W0x -> A -> B."""

    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        h = F.linear(x, self.base.weight)
        h = h + F.linear(F.linear(self.drop(h), self.A), self.B) * self.scale
        return h + self.base.bias if self.base.bias is not None else h


class IA3Out(nn.Module):
    """Liu et al. 2022 (IA3): a learned vector rescales a projection's output (keys / values). Starts at 1."""

    def __init__(self, base):
        super().__init__()
        self.base = base
        self.scale = nn.Parameter(torch.ones(base.out_features))

    def forward(self, x):
        return self.base(x) * self.scale


class IA3In(nn.Module):
    """IA3 on the feed-forward block: rescale the hidden activation just before the down projection."""

    def __init__(self, base):
        super().__init__()
        self.base = base
        self.scale = nn.Parameter(torch.ones(base.in_features))

    def forward(self, x):
        return self.base(x * self.scale)


def wrap_linears(root, names, make):
    swapped = 0
    for parent in list(root.modules()):
        for name, child in list(parent.named_children()):
            if name in names and isinstance(child, nn.Linear):
                setattr(parent, name, make(child))
                swapped += 1
    return swapped


# ------------------------------------------------------------------------------------------ CRF
class CRF(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.trans = nn.Parameter(torch.zeros(n, n))           # trans[prev, cur]
        self.start = nn.Parameter(torch.zeros(n))
        self.end = nn.Parameter(torch.zeros(n))

    def nll(self, em, tags):
        """Negative log-likelihood of one tag sequence. em: (T, n) emissions."""
        em = em.float()
        T = em.size(0)
        gold = self.start[tags[0]] + em[0, tags[0]] + self.end[tags[-1]]
        if T > 1:
            gold = gold + self.trans[tags[:-1], tags[1:]].sum() + em[torch.arange(1, T), tags[1:]].sum()
        alpha = self.start + em[0]
        for t in range(1, T):
            alpha = torch.logsumexp(alpha[:, None] + self.trans, dim=0) + em[t]
        return torch.logsumexp(alpha + self.end, dim=0) - gold

    @torch.no_grad()
    def viterbi(self, em):
        em = em.float().cpu().numpy()
        trans = self.trans.detach().float().cpu().numpy()
        score = self.start.detach().float().cpu().numpy() + em[0]
        back = np.zeros(em.shape, dtype=np.int64)
        for t in range(1, len(em)):
            cand = score[:, None] + trans
            back[t] = cand.argmax(0)
            score = cand.max(0) + em[t]
        score = score + self.end.detach().float().cpu().numpy()
        path = [int(score.argmax())]
        for t in range(len(em) - 1, 0, -1):
            path.append(int(back[t, path[-1]]))
        return path[::-1]


# ---------------------------------------------------------------------------------------- model
class RRModel(nn.Module):
    def __init__(self, backbone, cfg):
        super().__init__()
        self.backbone = backbone
        H = cfg.CONTEXT_DIM
        self.proj = nn.Sequential(nn.Linear(backbone.d_model, H), nn.LayerNorm(H))
        self.pos_emb = nn.Embedding(N_POS_BUCKETS, H)
        self.dom_emb = nn.Embedding(len(DOMAINS), H)
        self.drop = nn.Dropout(cfg.HEAD_DROPOUT)
        self.use_lstm = cfg.DOC_CONTEXT == "bilstm"
        self.lstm = nn.LSTM(H, H // 2, num_layers=2, bidirectional=True, batch_first=True, dropout=cfg.HEAD_DROPOUT)
        self.emit = nn.Linear(H, N_LABELS)
        self.shift = nn.Linear(2 * H, 1)
        self.crf = CRF(N_LABELS)
        self.prompt = None                      # (P, d_model) soft prompt, only for prompt tuning
        self.pack_tokens = cfg.PACK_TOKENS
        self.sentence_only = not cfg.PACK_CONTEXT

    def sentence_vectors(self, doc_tok, pad_id, device, pack_batch=None):
        """Encode a run of sentences (list of token lists) -> (S, d_model) mean-pooled sentence vectors."""
        packs, where = pack(doc_tok, self.pack_tokens)
        ids, mask, sot = batch_packs(packs, where, pad_id, device, self.sentence_only)
        S = len(doc_tok)
        out = torch.zeros(S, self.backbone.d_model, device=device)
        count = torch.zeros(S, 1, device=device)
        step = pack_batch or ids.size(0)
        for b in range(0, ids.size(0), step):
            h = self.backbone(ids[b:b + step], mask[b:b + step], prompt=self.prompt).float()
            s = sot[b:b + step].reshape(-1)
            keep = s >= 0
            out = out.index_add(0, s[keep], h.reshape(-1, h.size(-1))[keep])
            count = count.index_add(0, s[keep], torch.ones(int(keep.sum()), 1, device=device))
        return out / count.clamp(min=1)

    def head(self, sent_vecs, domain, n_total=None, offset=0):
        """Sentence vectors of a contiguous run -> (context states, emissions, shift logits)."""
        S = sent_vecs.size(0)
        n_total = n_total or S
        buckets = position_buckets(n_total)[offset:offset + S].to(sent_vecs.device)
        x = self.proj(sent_vecs) + self.pos_emb(buckets) + self.dom_emb.weight[domain]
        x = self.drop(x)
        if self.use_lstm:
            x, _ = self.lstm(x.unsqueeze(0))
            x = x.squeeze(0)
        ctx = self.drop(x)
        em = self.emit(ctx).float()
        shift = self.shift(torch.cat([ctx[:-1], ctx[1:]], dim=-1)).squeeze(-1).float() if S > 1 else em.new_zeros(0)
        return ctx, em, shift

    def forward(self, doc_tok, domain, pad_id, device, n_total=None, offset=0, pack_batch=None):
        vecs = self.sentence_vectors(doc_tok, pad_id, device, pack_batch)
        return self.head(vecs, domain, n_total, offset)


# ----------------------------------------------------------------------------- methods / params
def configure_method(model, method, cfg, tok_emb_weight):
    bb = model.backbone
    if method in ("full", "kd"):
        for p in model.parameters():
            p.requires_grad = True
        return "all weights trainable"

    for p in bb.parameters():
        p.requires_grad = False
    if method == "lora":
        n = wrap_linears(bb.layers, cfg.ADAPTER_TARGETS, lambda m: LoRALinear(m, cfg.ADAPTER_RANK, cfg.LORA_ALPHA, cfg.ADAPTER_DROPOUT))
        return f"LoRA on {n} projections (r={cfg.ADAPTER_RANK})"
    if method == "lorma":
        n = wrap_linears(bb.layers, cfg.ADAPTER_TARGETS, lambda m: LoRMALinear(m, cfg.ADAPTER_RANK, cfg.LORMA_ALPHA, cfg.ADAPTER_DROPOUT))
        return f"LoRMA on {n} projections (r={cfg.ADAPTER_RANK})"
    if method == "ia3":
        n_out = wrap_linears(bb.layers, cfg.IA3_OUT_TARGETS, IA3Out)
        n_in = wrap_linears(bb.layers, cfg.IA3_IN_TARGETS, IA3In)
        return f"IA3 vectors on {n_out} key/value projections and {n_in} feed-forward layers"
    if method == "prompt":
        # Lester et al. 2021: learned virtual tokens, initialised from real token embeddings
        rng = torch.Generator().manual_seed(cfg.SEED)
        pick = torch.randint(100, tok_emb_weight.size(0) - 100, (cfg.PROMPT_LEN,), generator=rng)
        model.prompt = nn.Parameter(tok_emb_weight[pick].detach().clone())
        return f"{cfg.PROMPT_LEN} soft-prompt vectors prepended to every pack"
    raise ValueError(f"unknown method {method}")


def count_params(model):
    every = sum(p.numel() for p in model.parameters())
    train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return every, train


def trainable_state(model):
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().float().half() for k, v in model.state_dict().items() if k in names}


def layer_depth(name, n_layers):
    if ".layers." in name:
        return int(name.split(".layers.")[1].split(".")[0]) + 1
    if "emb" in name and name.startswith("backbone."):
        return 0
    return n_layers + 1


def build_optimizer(model, method, cfg):
    n_layers = len(model.backbone.layers)
    groups = {}
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if not name.startswith("backbone.") and name != "prompt":
            lr = cfg.HEAD_LR                                    # BiLSTM / CRF / embeddings of the task head
        elif name == "prompt":
            lr = cfg.PROMPT_LR
        elif method in ("full", "kd"):
            lr = cfg.FULL_LR * cfg.LLRD ** (n_layers + 1 - layer_depth(name, n_layers))
        else:
            lr = cfg.ADAPTER_LR
        wd = 0.0 if (p.ndim < 2 or name == "prompt") else cfg.WEIGHT_DECAY
        groups.setdefault((lr, wd), []).append(p)
    return torch.optim.AdamW([{"params": ps, "lr": lr, "weight_decay": wd} for (lr, wd), ps in groups.items()],
                             betas=(0.9, 0.98), eps=1e-6)


def class_weights(docs, power):
    counts = np.ones(N_LABELS)
    for d in docs:
        counts += np.bincount(d["labels"], minlength=N_LABELS)
    w = (counts.sum() / counts) ** power
    return torch.tensor(w / w.mean(), dtype=torch.float32), torch.tensor(np.log(counts / counts.sum()), dtype=torch.float32)


# --------------------------------------------------------------------------------------- metric
def macro_scores(gold, pred):
    """Macro P / R / F1 over the labels that occur in gold or prediction (sklearn's default behaviour)."""
    gold, pred = np.asarray(gold), np.asarray(pred)
    per = {}
    for lab in sorted(set(gold.tolist()) | set(pred.tolist())):
        tp = int(((gold == lab) & (pred == lab)).sum())
        p = tp / max(1, int((pred == lab).sum()))
        r = tp / max(1, int((gold == lab).sum()))
        per[LABELS[lab]] = {"p": p, "r": r, "f1": 2 * p * r / (p + r) if p + r else 0.0, "gold": int((gold == lab).sum())}
    macro = {k: float(np.mean([v[k] for v in per.values()])) for k in ("p", "r", "f1")}
    return macro, per


def top_confusions(gold, pred, k=8):
    pairs = {}
    for g, p in zip(gold, pred):
        if g != p:
            pairs[(LABELS[g], LABELS[p])] = pairs.get((LABELS[g], LABELS[p]), 0) + 1
    return sorted(pairs.items(), key=lambda kv: -kv[1])[:k]


# ------------------------------------------------------------------------------------ inference
@torch.no_grad()
def document_emissions(model, docs, pad_id, device, pack_batch):
    model.eval()
    out = []
    for d in docs:
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            _, em, _ = model(d["tok"], d["domain"], pad_id, device, pack_batch=pack_batch)
        out.append(F.log_softmax(em.float(), -1))
    return out


def decode(model, emissions, log_prior, decoder, tau):
    preds = []
    for em in emissions:
        adj = em - tau * log_prior.to(em.device)                # logit adjustment (Menon et al. 2021) for macro-F1
        preds.append(model.crf.viterbi(adj) if decoder == "crf" else adj.argmax(-1).tolist())
    return preds


def tune_decoding(model, docs, emissions, log_prior, taus):
    """Pick decoder (CRF / argmax) and prior-correction strength on the dev documents."""
    gold = [l for d in docs for l in d["labels"]]
    best = None
    grid = []
    for decoder in ("crf", "argmax"):
        for tau in taus:
            pred = [l for p in decode(model, emissions, log_prior, decoder, tau) for l in p]
            f1 = macro_scores(gold, pred)[0]["f1"]
            grid.append({"decoder": decoder, "tau": tau, "f1": f1})
            if best is None or f1 > best["f1"]:
                best = {"decoder": decoder, "tau": tau, "f1": f1}
    return best, grid


def write_submission(test_docs, preds, out_dir, zip_name):
    os.makedirs(out_dir, exist_ok=True)
    payload = {}
    for d, p in zip(test_docs, preds):
        assert len(p) == len(d["sents"]), f"{d['id']}: {len(p)} predictions for {len(d['sents'])} sentences"
        payload[d["id"]] = [LABELS[i] for i in p]
    path = os.path.join(out_dir, "prediction.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False)
    zip_path = os.path.join(out_dir, zip_name)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(path, arcname="prediction.json")
    with zipfile.ZipFile(zip_path) as z:
        assert z.namelist() == ["prediction.json"]
    return path, zip_path


def training_spans(docs, span, rng):
    """Cut every document into runs of `span` consecutive sentences, starting at a random offset each epoch."""
    spans = []
    for di, d in enumerate(docs):
        n = len(d["sents"])
        if n <= span:
            spans.append((di, 0, n))
            continue
        start = rng.randrange(0, span)
        cuts = [0] + list(range(start, n, span))
        cuts = sorted(set(c for c in cuts if c < n)) + [n]
        for a, b in zip(cuts[:-1], cuts[1:]):
            if b - a >= 2:
                spans.append((di, a, b))
    rng.shuffle(spans)
    return spans

In [ ]:
%%writefile /kaggle/working/phase3_arch1/run_rr.py
"""
One Phase-3 experiment = one call:

    python run_rr.py --method lorma

Trains on CL + IT together (a domain embedding tells them apart), keeps the epoch with the best dev macro-F1,
tunes the decoder (CRF vs argmax) and the prior correction on dev, writes the test predictions + submission zip
and pushes everything to Hugging Face.
"""
import os
import sys
import json
import time
import random
import argparse

import numpy as np
import torch
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import config as C
import backbone as BB
from rr_core import (load_split, tokenize_docs, RRModel, configure_method, count_params, trainable_state,
                     build_optimizer, class_weights, macro_scores, top_confusions, document_emissions, decode,
                     tune_decoding, write_submission, training_spans, DOMAINS, LABELS)


def log(*a):
    print(*a, flush=True)


def build_model(method, keep_layers=None):
    model = RRModel(BB.build_backbone(keep_layers=keep_layers, bidirectional=C.BIDIRECTIONAL), C)
    note = configure_method(model, method, C, model.backbone.tok_emb.weight)
    return model, note


def load_finetuned(method, device):
    """Rebuild a model trained earlier by this notebook (the distillation teacher)."""
    path = os.path.join(C.OUT_DIR, method, f"rr_{method}.pt")
    if not os.path.exists(path):
        path = BB.fetch_from_hub(C.RR_REPO, f"{method}/rr_{method}.pt", os.path.join(C.OUT_DIR, "_hub"))
    if path is None or not os.path.exists(path):
        raise FileNotFoundError(f"No fine-tuned '{method}' model locally or on {C.RR_REPO}; run '{method}' first.")
    blob = torch.load(path, map_location="cpu")
    model, _ = build_model(method, blob.get("keep_layers"))
    _, unexpected = model.load_state_dict({k: v.float() for k, v in blob["weights"].items()}, strict=False)
    assert not unexpected, unexpected
    return model.to(device).eval()


def push(files, message):
    if not os.environ.get("HF_TOKEN"):
        return
    try:
        from huggingface_hub import HfApi
        api = HfApi()
        api.create_repo(C.RR_REPO, private=True, exist_ok=True, token=os.environ["HF_TOKEN"])
        for local, remote in files:
            api.upload_file(path_or_fileobj=local, path_in_repo=remote, repo_id=C.RR_REPO,
                            token=os.environ["HF_TOKEN"], commit_message=message)
        log(f"pushed to {C.RR_REPO}: {message}")
    except Exception as e:
        log(f"HF push failed ({type(e).__name__}: {e}) - the files are still in {C.OUT_DIR}")


def evaluate(model, docs, pad_id, device, log_prior):
    ems = document_emissions(model, docs, pad_id, device, C.EVAL_PACK_BATCH)
    gold = [l for d in docs for l in d["labels"]]
    pred = [l for p in decode(model, ems, log_prior, "crf", 0.0) for l in p]
    return macro_scores(gold, pred)[0]["f1"], ems


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--method", required=True, choices=["full", "lora", "lorma", "ia3", "prompt", "kd"])
    method = ap.parse_args().method

    random.seed(C.SEED)
    np.random.seed(C.SEED)
    torch.manual_seed(C.SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out_dir = os.path.join(C.OUT_DIR, method)
    os.makedirs(out_dir, exist_ok=True)
    t0 = time.time()

    # ---------------------------------------------------------------- data
    tok = BB.load_tokenizer()
    train = [d for dom in DOMAINS for d in load_split(C.FILES[f"{dom}_train"], dom)]
    dev = [d for dom in DOMAINS for d in load_split(C.FILES[f"{dom}_dev"], dom)]
    test = [d for dom in DOMAINS for d in load_split(C.FILES[f"{dom}_test"], dom)]
    if C.TRAIN_ON_DEV:
        train = train + dev                                     # final run only: dev is then no longer a fair check
    if C.TRAIN_DOC_LIMIT:
        train = train[:C.TRAIN_DOC_LIMIT // 2] + train[-(C.TRAIN_DOC_LIMIT // 2):]
    for docs in (train, dev, test):
        tokenize_docs(docs, tok, C.MAX_SENT_TOKENS)
    log(f"[{method}] train docs {len(train)} ({sum(len(d['sents']) for d in train):,} sentences) | "
        f"dev docs {len(dev)} | test docs {len(test)}")

    weights, log_prior = class_weights(train, C.CLASS_WEIGHT_POWER)
    weights = weights.to(device)

    # --------------------------------------------------------------- model
    keep = BB.student_layers() if method == "kd" else None
    model, note = build_model(method, keep)
    model.to(device)
    total, trainable = count_params(model)
    log(f"[{method}] {note} | params {total / 1e6:.1f}M, trainable {trainable / 1e6:.2f}M ({100 * trainable / total:.2f}%)")

    teacher = None
    if method == "kd":
        teacher = load_finetuned(C.KD_TEACHER, device)
        log(f"[kd] teacher = fine-tuned '{C.KD_TEACHER}' ({count_params(teacher)[0] / 1e6:.1f}M) -> "
            f"student with backbone layers {keep} ({total / 1e6:.1f}M)")

    optimizer = build_optimizer(model, method, C)
    base_lrs = [g["lr"] for g in optimizer.param_groups]
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    rng = random.Random(C.SEED)
    n_spans = len(training_spans(train, C.SPAN_SENTS, random.Random(0)))
    total_steps = -(-n_spans // C.GRAD_ACCUM) * C.EPOCHS
    warmup = max(1, int(C.WARMUP_FRAC * total_steps))

    best, stale, step, history = {"f1": -1.0}, 0, 0, []
    for epoch in range(1, C.EPOCHS + 1):
        model.train()
        spans = training_spans(train, C.SPAN_SENTS, rng)
        running, seen = 0.0, 0
        for i, (di, a, b) in enumerate(spans):
            d = train[di]
            gold = torch.tensor(d["labels"][a:b], device=device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
                ctx, em, shift = model(d["tok"][a:b], d["domain"], tok.pad_id, device, n_total=len(d["sents"]), offset=a)
                loss = model.crf.nll(em, gold) / len(gold)
                loss = loss + C.CE_WEIGHT * F.cross_entropy(em, gold, weight=weights)
                if len(gold) > 1:
                    changed = (gold[1:] != gold[:-1]).float()
                    loss = loss + C.SHIFT_WEIGHT * F.binary_cross_entropy_with_logits(shift, changed)
                if teacher is not None:
                    with torch.no_grad():
                        t_ctx, t_em, _ = teacher(d["tok"][a:b], d["domain"], tok.pad_id, device, n_total=len(d["sents"]), offset=a)
                    T = C.KD_TEMPERATURE
                    kl = F.kl_div(F.log_softmax(em / T, -1), F.softmax(t_em.float() / T, -1), reduction="batchmean") * T * T
                    hid = F.mse_loss(ctx.float(), t_ctx.float())     # match the teacher's sentence representations too
                    loss = C.KD_ALPHA * loss + (1 - C.KD_ALPHA) * kl + C.KD_HIDDEN_WEIGHT * hid
            scaler.scale(loss / C.GRAD_ACCUM).backward()
            running += loss.item()
            seen += 1

            if seen % C.GRAD_ACCUM == 0 or i == len(spans) - 1:
                frac = (step + 1) / warmup if step < warmup else max(0.0, (total_steps - step) / max(1, total_steps - warmup))
                for g, lr in zip(optimizer.param_groups, base_lrs):
                    g["lr"] = lr * frac
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                step += 1

        f1, _ = evaluate(model, dev, tok.pad_id, device, log_prior)
        history.append({"epoch": epoch, "train_loss": running / max(1, seen), "dev_macro_f1": f1})
        log(f"[{method}] epoch {epoch}/{C.EPOCHS} | loss {running / max(1, seen):.4f} | dev macro-F1 (CRF) {f1:.4f} | "
            f"{(time.time() - t0) / 60:.1f} min")
        if f1 > best["f1"]:
            best = {"f1": f1, "epoch": epoch}
            torch.save({"method": method, "keep_layers": keep, "weights": trainable_state(model), "dev_f1": f1,
                        "epoch": epoch}, os.path.join(out_dir, f"rr_{method}.pt"))
            stale = 0
        else:
            stale += 1
            if stale >= C.PATIENCE:
                log(f"[{method}] no dev improvement for {C.PATIENCE} epochs - stopping early")
                break

    # ------------------------------------------- best epoch: tune decoding on dev, predict test
    blob = torch.load(os.path.join(out_dir, f"rr_{method}.pt"), map_location="cpu")
    model.load_state_dict({k: v.float() for k, v in blob["weights"].items()}, strict=False)
    t_inf = time.time()
    dev_ems = document_emissions(model, dev, tok.pad_id, device, C.EVAL_PACK_BATCH)
    dev_seconds = time.time() - t_inf
    choice, grid = tune_decoding(model, dev, dev_ems, log_prior, C.TAU_GRID)
    dev_pred = decode(model, dev_ems, log_prior, choice["decoder"], choice["tau"])

    gold_all = [l for d in dev for l in d["labels"]]
    pred_all = [l for p in dev_pred for l in p]
    macro, per = macro_scores(gold_all, pred_all)
    by_domain = {}
    for k, dom in enumerate(DOMAINS):
        g = [l for d in dev if d["domain"] == k for l in d["labels"]]
        p = [l for d, pr in zip(dev, dev_pred) if d["domain"] == k for l in pr]
        by_domain[dom] = macro_scores(g, p)[0]

    test_ems = document_emissions(model, test, tok.pad_id, device, C.EVAL_PACK_BATCH)
    test_pred = decode(model, test_ems, log_prior, choice["decoder"], choice["tau"])
    _, zip_path = write_submission(test, test_pred, out_dir, f"submission_{C.ARCH}_{method}.zip")

    summary = {
        "arch": C.ARCH, "method": method, "best_epoch": best["epoch"], "decoder": choice["decoder"], "tau": choice["tau"],
        "dev_macro": macro, "dev_by_domain": by_domain, "dev_per_label": per, "decoding_grid": grid,
        "confusions": [[g, p, n] for (g, p), n in top_confusions(gold_all, pred_all)],
        "params_total": total, "params_trainable": trainable, "dev_inference_seconds": round(dev_seconds, 2),
        "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if device.type == "cuda" else 0.0,
        "train_minutes": round((time.time() - t0) / 60, 1), "history": history,
        "trained_on_dev": bool(C.TRAIN_ON_DEV),
    }
    with open(os.path.join(out_dir, "results.json"), "w") as f:
        json.dump(summary, f, indent=2)
    log(f"[{method}] DONE | dev macro-F1 {macro['f1']:.4f} (CL {by_domain['CL']['f1']:.4f}, IT {by_domain['IT']['f1']:.4f}) "
        f"| decoder {choice['decoder']}, tau {choice['tau']} | epoch {best['epoch']} -> {zip_path}")

    push([(os.path.join(out_dir, f"rr_{method}.pt"), f"{method}/rr_{method}.pt"),
          (os.path.join(out_dir, "results.json"), f"{method}/results.json"),
          (zip_path, f"{method}/{os.path.basename(zip_path)}")], f"{C.ARCH} {method}: dev macro-F1 {macro['f1']:.4f}")


if __name__ == "__main__":
    main()

## 4. Phase-3 data

In [ ]:
import importlib
import config as C
importlib.reload(C)

for dom in ("CL", "IT"):
    for name in (f"Train/{dom}_train.json", f"Train/{dom}_dev.json", f"Test/{dom}_test.json"):
        hf_hub_download(C.HF_DATASET_REPO, f"Phase_3_RR/{name}", repo_type="dataset", local_dir=C.DATA_DIR, token=HF_TOKEN)
hf_hub_download(C.HF_DATASET_REPO, "Phase_3_RR/Train/label_vocab.json", repo_type="dataset", local_dir=C.DATA_DIR, token=HF_TOKEN)

import rr_core as R
importlib.reload(R)
vocab = json.load(open(os.path.join(C.DATA_DIR, "Phase_3_RR/Train/label_vocab.json")))
assert sorted(vocab, key=vocab.get) == R.LABELS, "label vocabulary changed - update LABELS in rr_core.py"

from collections import Counter
for dom in R.DOMAINS:
    tr, dv, te = (R.load_split(C.FILES[f"{dom}_{s}"], dom) for s in ("train", "dev", "test"))
    counts = Counter(R.LABELS[l] for d in tr for l in d["labels"])
    sticky = sum(a == b for d in tr for a, b in zip(d["labels"], d["labels"][1:])) / sum(len(d["labels"]) - 1 for d in tr)
    print(f"{dom}: train {len(tr)} docs / {sum(len(d['sents']) for d in tr):,} sentences | dev {len(dv)} | test {len(te)} docs "
          f"| same role as previous sentence: {sticky:.0%}")
    print("   ", dict(counts.most_common()))

## 5. Load the Phase-1 weights once
This checks the model and caches the files for the experiment processes.

In [ ]:
import backbone as BB
importlib.reload(BB)
tok = BB.load_tokenizer()
probe = BB.build_backbone()
sample = R.tokenize_docs(R.load_split(C.FILES["CL_dev"], "CL")[:1], tok, C.MAX_SENT_TOKENS)[0]
packs, where = R.pack(sample["tok"][:40], C.PACK_TOKENS)
ids, mask, _ = R.batch_packs(packs, where, tok.pad_id, torch.device("cpu"))
with torch.no_grad():
    h = probe(ids, mask)
print(f"backbone ready: {sum(p.numel() for p in probe.parameters()) / 1e6:.1f}M parameters | "
      f"40 sentences -> {len(packs)} packs -> hidden states {tuple(h.shape)}")
del probe, h

## 6. Run the experiments
Each experiment is its own process (`run_rr.py --method ...`); with two GPUs, two run at once.
`kd` starts as soon as `full` (its teacher) is done. Results, checkpoints and zips are pushed to your HF repo as they finish.

In [ ]:
import subprocess, threading

pending = list(C.METHODS)
done, failed = set(), set()
lock = threading.Lock()


def next_job():
    with lock:
        if "kd" in pending and C.KD_TEACHER in failed:
            pending.remove("kd")
            print(f"skipping kd: its teacher '{C.KD_TEACHER}' failed")
        for m in pending:
            if m != "kd" or C.KD_TEACHER in done or C.KD_TEACHER not in C.METHODS:
                pending.remove(m)
                return m
        return "wait" if pending else None


def worker(gpu):
    while True:
        job = next_job()
        if job is None:
            return
        if job == "wait":
            time.sleep(15)
            continue
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
        proc = subprocess.Popen([sys.executable, "run_rr.py", "--method", job], cwd=WORK_DIR, env=env,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(f"[gpu{gpu}] {line}", end="")
        with lock:
            (done if proc.wait() == 0 else failed).add(job)


t0 = time.time()
threads = [threading.Thread(target=worker, args=(g,)) for g in range(max(1, torch.cuda.device_count()))]
for t in threads:
    t.start()
for t in threads:
    t.join()
print(f"\nfinished in {(time.time() - t0) / 60:.0f} min | ok: {sorted(done)} | failed: {sorted(failed)}")

## 7. Results
Dev macro-F1 overall and per domain, cost of each method, per-role scores and the most frequent confusions.

In [ ]:
import pandas as pd

rows, per_label, confusions = [], {}, {}
for path in sorted(glob.glob(os.path.join(C.OUT_DIR, "*", "results.json"))):
    r = json.load(open(path))
    m = r["method"]
    rows.append({"method": m, "dev macro-F1": r["dev_macro"]["f1"], "CL F1": r["dev_by_domain"]["CL"]["f1"],
                 "IT F1": r["dev_by_domain"]["IT"]["f1"], "macro P": r["dev_macro"]["p"], "macro R": r["dev_macro"]["r"],
                 "decoder": r["decoder"], "tau": r["tau"], "best epoch": r["best_epoch"],
                 "params (M)": r["params_total"] / 1e6, "trainable %": 100 * r["params_trainable"] / r["params_total"],
                 "train min": r["train_minutes"], "peak GPU GB": r["peak_gpu_gb"], "dev infer s": r["dev_inference_seconds"]})
    per_label[m] = {lab: v["f1"] for lab, v in r["dev_per_label"].items()}
    confusions[m] = r["confusions"]

table = pd.DataFrame(rows).sort_values("dev macro-F1", ascending=False).round(4)
display(table)
display(pd.DataFrame(per_label).round(3))
best = table.iloc[0]["method"]
print(f"most frequent confusions for '{best}' (gold -> predicted, count):")
for g, p, n in confusions[best]:
    print(f"   {g:24s} -> {p:24s} {n}")

## 8. Pick a submission
Every experiment wrote `runs/<method>/submission_<arch>_<method>.zip`: a single `prediction.json` at the root with one
label per sentence for all CL and IT test documents. The best on dev is also copied to `/kaggle/working/submission.zip`.

In [ ]:
src = glob.glob(os.path.join(C.OUT_DIR, best, "submission_*.zip"))[0]
shutil.copy(src, "/kaggle/working/submission.zip")
print(f"best on dev: {best} -> /kaggle/working/submission.zip")
for z in sorted(glob.glob(os.path.join(C.OUT_DIR, "*", "submission_*.zip"))):
    print("  ", z)